Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Downloading many data series with a loop

In Week 4, we downloaded **one** series from an API. Here we use a **loop** to download **six series of your choice**, saving each one as a JSON file, ready to chart on your website.

This is portfolio task **CC6** (part 1). A shorter version, without the explanations, is in **DS5_2_Loop_API_Download_Clean**.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
import requests   # fetch things from the internet

<br><br>

---

<br>

## 1. Get one series working first

Before writing a loop, always get the steps working for **one** item. Then the loop just repeats them.

We use the [FRED API](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) (US Federal Reserve data). FRED needs an **API key**: a password that identifies who is asking. Register for your own key [here](https://fredaccount.stlouisfed.org/login/secure/).

In [ ]:
api_key = "22ee7a76e736e32f54f5df0a7171538d"   # the course key (see the note below)

> 🔑 **API keys are like passwords: they identify you.** We share this course key so the demo works for everyone. In your own work:
> - Get your own free key: [fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html)
> - **Free** keys that anyone can sign up for, like FRED's, are low risk. It's fine for one to appear in a public chart (as in our live Vega-Lite examples): at worst, someone uses up your request limit and you make a new key.
> - **Paid or private** keys (e.g. for commercial data, AI models or cloud services) should be kept out of public GitHub repositories, websites and charts: anyone could copy and use them, and you might pay the bill. (GitHub will often warn you if it spots one.)
> - To keep a key out of your code: in Colab, use **Secrets** (the 🔑 icon on the left); on your own computer, use a **`.env`** file. Both are shown in the optional cells at the end of this notebook.
> - If a key is leaked, delete it on the provider's website and make a new one.

<br>

Now build the URL for one series, unemployment (`UNRATE`), and request it:

In [1]:
series_id = "UNRATE"

url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"

response = requests.get(url)
response

NameError: name 'api_key' is not defined

`<Response [200]>` means it worked. Let's look at the data inside:

In [2]:
data = response.json()

data['observations'][:3]   # the first 3 observations

NameError: name 'response' is not defined

<br>

Two things to notice (useful when we chart it later):
- The data is inside `observations`, so a Vega-Lite chart needs `"format": {"property": "observations"}`, just like `"[1]"` for the World Bank.
- The values are **text** (`'3.4'`), and FRED writes missing values as `'.'`. Vega-Lite turns the text into numbers for us.

<br><br>

---

<br>

## 2. Save it as a file

We save exactly what FRED sent us, as a JSON file. `open(..., 'w')` creates a file to **w**rite into, and `.write()` puts the text in it:

> **What's `encoding="utf-8"`?** It tells Python how to store letters and symbols in the file. Colab doesn't need it, but on Windows Python uses an older encoding by default, which can fail on characters like `ô` or `ã` (e.g. "Côte d'Ivoire" or "São Tomé" in World Bank data). Adding it makes the code work the same on every computer.

In [3]:
filename = f"data_US_FRED-{series_id}.json"

with open(filename, "w", encoding="utf-8") as file:
    file.write(response.text)

print(f"Saved {filename}")

NameError: name 'response' is not defined

The file appears in Colab's **Files** panel (the folder icon on the left).

<br><br>

---

<br>

## 3. The loop: six series of your choice

Now we put those steps inside a loop. The **only** thing that changes each time is `series_id`.

**Choose six series** and put their codes in the list `fred_series` below, in quotes and separated by commas. Find codes on [fred.stlouisfed.org](https://fred.stlouisfed.org/): the code is at the end of each series' web address (e.g. fred.stlouisfed.org/series/**UNRATE**).

Some examples to get you started:

| Code | Series | How often |
|---|---|---|
| `GDPC1` | Real GDP (US output, adjusted for inflation) | quarterly |
| `UNRATE` | Unemployment rate, % | monthly |
| `CPIAUCSL` | Consumer price index (the price level, used to measure inflation) | monthly |
| `FEDFUNDS` | Federal funds rate: the US central bank's main interest rate, % | monthly |
| `PAYEMS` | Number of jobs (non-farm payroll employment), thousands | monthly |
| `INDPRO` | Industrial production index (output of factories, mines and utilities) | monthly |

> Monthly or quarterly series are best: **daily** series (e.g. exchange rates) make very large files.

In [ ]:
fred_series = []      # put your six series codes here, e.g. ["UNRATE", "CPIAUCSL", ...]

for series_id in fred_series:
    print(f"--- Series: {series_id} ---")

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    print("Requesting data from FRED...")
    response = requests.get(url)
    print(f"Response: {response.status_code}")

    filename = f"data_US_FRED-{series_id}.json"
    with open(filename, "w", encoding="utf-8") as file:
        file.write(response.text)
    print(f"Saved as {filename}")
    print()      # a blank line between series

print("Finished: the loop has gone through every series in the list.")

<br>

Compare the loop with sections 1 and 2: it's the same lines, indented, with `series_id` changing each time. The `print()` lines tell us what is happening on each **iteration** (each time round the loop). If the list is still empty, the loop runs zero times and only the last line prints.

> **Tip: building a loop.** Get it working for one item first. Then try the loop with **two** items, and only then the full list. `print()` inside the loop shows you how far it got.

<br><br>

---

<br>

## 4. Add a check: what if something goes wrong?

What if a series code is wrong? FRED replies with an error instead of data, and our loop would happily save the error message as if it were data.

A **conditional** fixes this: only save the file **if** the request worked. `200` is the web's code for "OK". Notice the deliberate mistake in the list:

In [ ]:
fred_series_test = ["UNRATE", "NOT_A_SERIES", "CPIAUCSL"]

for series_id in fred_series_test:
    print(f"--- Series: {series_id} ---")

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    print(f"Response: {response.status_code}")

    if response.status_code == 200:
        filename = f"data_US_FRED-{series_id}.json"
        with open(filename, "w", encoding="utf-8") as file:
            file.write(response.text)
        print(f"OK: saved as {filename}")
    else:
        print(f"Problem with {series_id}: nothing saved")
    print()

The loop carries on after the problem, and tells us which series failed. (We used a new list name, `fred_series_test`, so your own list from section 3 is not overwritten.)

Now add the check to your own loop: run the cell below, which uses your list from section 3.

In [ ]:
for series_id in fred_series:
    print(f"--- Series: {series_id} ---")

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    print(f"Response: {response.status_code}")

    if response.status_code == 200:
        filename = f"data_US_FRED-{series_id}.json"
        with open(filename, "w", encoding="utf-8") as file:
            file.write(response.text)
        print(f"OK: saved as {filename}")
    else:
        print(f"Problem with {series_id}: nothing saved")
    print()

<br><br>

---

<br>

## 5. Download the files

The files are only stored in Colab for now. This loop downloads each one to your computer.

> Your browser may ask whether to allow **multiple downloads**: click **Allow**.

In [ ]:
from google.colab import files     # only exists in Colab

for series_id in fred_series:
    filename = f"data_US_FRED-{series_id}.json"
    print(f"Downloading {filename}")
    files.download(filename)

<br>

### Next steps

1. Put your six JSON files in your website folder (e.g. a `data` folder), commit and sync them to GitHub.
2. In the seminar, we use a **JavaScript** loop to make a chart for each file: a mini dashboard.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Find two more series on FRED that interest you. Add their codes to your list and run the loop again.

**2.** Change the loop to download data from the **World Bank** API (Week 4) for several countries. Hint: the country code goes in the URL, and no key is needed.

In [ ]:
# Your code here


<br><br>

---

<br>

## Annex: the same loop with a `.format()` template

Some people prefer to write the URL **once**, as a template with `{ }` placeholders, before the loop (see the annex in DS5_1). `.format()` then fills in the gaps, in order:

In [ ]:
url_template = "https://api.stlouisfed.org/fred/series/observations?series_id={}&api_key={}&file_type=json"
filename_template = "data_US_FRED-{}.json"

for series_id in fred_series:
    print(f"--- Series: {series_id} ---")
    url = url_template.format(series_id, api_key)
    response = requests.get(url)

    if response.status_code == 200:
        filename = filename_template.format(series_id)
        with open(filename, "w", encoding="utf-8") as file:
            file.write(response.text)
        print(f"OK: saved as {filename}")
    else:
        print(f"Problem with {series_id}: error {response.status_code}")
    print()

<br><br>

---

<br>

## Optional: keeping your key secret in Colab

1. Click the 🔑 **Secrets** icon in Colab's left sidebar.
2. Click **Add new secret**. Name it `FRED_API_KEY`, paste your key as the value, and switch on **Notebook access**.
3. Run the cell below. Your key is now used in the code, but never written in the notebook, so it's safe to share.

In [ ]:
from google.colab import userdata

api_key = userdata.get('FRED_API_KEY')

<br>

## Optional: keeping your key secret on your own computer (`.env` file)

When you run notebooks on your own computer (e.g. in VS Code), the usual way is a **`.env`** file: a small text file, kept next to your notebook, that holds your secrets.

1. In your project folder, create a file called **`.env`** (the dot at the start matters) containing one line:<br>`FRED_API_KEY=your_key_here`
2. Check that **`.env`** is listed in your **`.gitignore`** file, so it is never uploaded to GitHub. (The course template's `.gitignore` already does this.)
3. Install the package that reads it (once): `pip install python-dotenv`
4. Run the cell below. Like Colab Secrets, the key is used but never written in the notebook.

In [ ]:
from dotenv import load_dotenv     # reads .env files (pip install python-dotenv)
import os

load_dotenv()                          # finds the .env file and loads what's in it
api_key = os.getenv("FRED_API_KEY")    # get the value called FRED_API_KEY

print("Key found" if api_key else "No key found: check your .env file")